# Attention Lab 6 · Multi-Head, MQA & GQA

> **Types of Attention — Lab 6.** Run top to bottom (Runtime → Run all). Read the plain-English notes, run the code,
> then do the **Your turn 🧪** cell. Everything is built from scratch with NumPy so nothing is hidden.

### In plain English
One focuser notices one kind of link. **Multi-head attention** runs a **team** of focusers in
parallel, each trained to spot something different — nearby grammar, who *"it"* refers to, verb-and-object pairs — like
a grammar expert, a character-tracker and an action-tracker all reading the same sentence. Their findings get combined.

**The catch:** as a model writes, each focuser keeps notes on everything so far (the *KV cache*). Many focusers = a huge
pile of notes = slow generation. The fix is to **share notebooks**:
- **MQA** — all focusers share **one** notebook. Lightest and fastest, small quality cost.
- **GQA** — focusers share in a **few small groups**. Almost full quality, most of the speed. Used by Llama.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
VIOLET, CYAN = "#6C5CE7", "#22D3EE"
rng = np.random.default_rng(0)
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True); e = np.exp(x); return e / e.sum(axis=axis, keepdims=True)

## 1 · Multi-head attention from scratch
Project into `h` smaller Q/K/V sets, attend in each head, concatenate, mix with an output layer.

In [ ]:
def multi_head_attention(X, W_q, W_k, W_v, W_o, h):
    n, d_model = X.shape
    d_head = d_model // h
    Q = (X @ W_q).reshape(n, h, d_head).transpose(1,0,2)   # (h, n, d_head)
    K = (X @ W_k).reshape(n, h, d_head).transpose(1,0,2)
    V = (X @ W_v).reshape(n, h, d_head).transpose(1,0,2)
    scores = Q @ K.transpose(0,2,1) / np.sqrt(d_head)      # (h, n, n)
    W = softmax(scores)
    heads = W @ V                                          # (h, n, d_head)
    concat = heads.transpose(1,0,2).reshape(n, d_model)    # glue heads back together
    return concat @ W_o, W

n, d_model, h = 8, 32, 4
X = rng.normal(size=(n, d_model))
W_q, W_k, W_v, W_o = (rng.normal(size=(d_model,d_model))*0.2 for _ in range(4))
out, W = multi_head_attention(X, W_q, W_k, W_v, W_o, h)
print("output", out.shape, "| per-head attention", W.shape, "(heads, queries, keys)")
print("each head's rows sum to 1:", np.allclose(W.sum(-1), 1))

## 2 · Heads specialize
Real heads learn different jobs. We'll *hand-build* three so you can see the idea clearly.

In [ ]:
sent = ["the","dog","saw","the","cat","and","it","barked"]
is_noun = np.array([0,1,0,0,1,0,0,0]); is_verb = np.array([0,0,1,0,0,0,0,1]); is_pron = np.array([0,0,0,0,0,0,1,0])
N = len(sent); i, j = np.indices((N,N))

near = -np.abs(i-j) * 0.35            # mild "prefer the closer word" tiebreak

head_local  = -np.abs(i-j) * 1.6                                   # look at neighbours
head_coref  = np.outer(is_pron, is_noun) * 3.0 + near + 0.15       # pronoun -> nearest noun
head_pred   = (np.outer(is_verb, is_noun) + np.outer(is_noun, is_verb)) * 3.0 + near + 0.15   # verb <-> noun

for name, S in [("local", head_local), ("coreference", head_coref), ("predicate", head_pred)]:
    W_h = softmax(S)
    q = sent.index("it") if name=="coreference" else sent.index("barked") if name=="predicate" else 4
    top = np.argsort(-W_h[q])[:2]
    print(f'{name:12s}: "{sent[q]}" -> ' + ", ".join(f"{sent[t]} ({W_h[q,t]*100:.0f}%)" for t in top))

## 3 · MHA vs GQA vs MQA — the KV-cache saving
Every query head needs a Key/Value head. Sharing them shrinks the cache the model must hold while generating.

In [ ]:
def kv_cache_bytes(seq_len, n_kv_heads, d_head, n_layers=32, bytes_per=2):
    # keys + values, per layer
    return 2 * seq_len * n_kv_heads * d_head * n_layers * bytes_per

seq_len, d_head, n_q_heads = 4096, 128, 32
for name, n_kv in [("MHA  (32 KV heads)", 32), ("GQA  (8 KV groups)", 8), ("MQA  (1 KV head)", 1)]:
    b = kv_cache_bytes(seq_len, n_kv, d_head)
    print(f"{name:22s} KV cache = {b/1e9:6.2f} GB   ({32/n_kv:.0f}x smaller)")

In [ ]:
# who shares with whom?
def kv_index(q_head, n_q, n_kv): return q_head // (n_q // n_kv)
fig, axes = plt.subplots(1, 3, figsize=(12,3.2))
for ax, (name, n_kv) in zip(axes, [("MHA",8), ("GQA",2), ("MQA",1)]):
    for qh in range(8):
        kv = kv_index(qh, 8, n_kv)
        ax.plot([qh, (kv+.5)*(8/n_kv)-.5], [1, 0], color=VIOLET, alpha=.6)
    ax.scatter(range(8), [1]*8, s=180, color=VIOLET, zorder=3, label="query heads")
    ax.scatter([(k+.5)*(8/n_kv)-.5 for k in range(n_kv)], [0]*n_kv, s=220, color=CYAN, zorder=3, label="KV heads")
    ax.set_title(f"{name}: 8 query -> {n_kv} KV"); ax.axis("off")
axes[0].legend(loc="center left", fontsize=8)
plt.tight_layout(); plt.show()

## 4 · Implement GQA
Query heads are grouped; each group reuses one Key/Value head via `np.repeat`.

In [ ]:
def grouped_query_attention(X, n_q_heads, n_kv_heads, d_model, seed=0):
    rng = np.random.default_rng(seed)
    n = X.shape[0]; d_head = d_model // n_q_heads
    Q = (X @ rng.normal(size=(d_model, n_q_heads*d_head))*0.2).reshape(n, n_q_heads, d_head).transpose(1,0,2)
    K = (X @ rng.normal(size=(d_model, n_kv_heads*d_head))*0.2).reshape(n, n_kv_heads, d_head).transpose(1,0,2)
    V = (X @ rng.normal(size=(d_model, n_kv_heads*d_head))*0.2).reshape(n, n_kv_heads, d_head).transpose(1,0,2)
    reps = n_q_heads // n_kv_heads
    K = np.repeat(K, reps, axis=0)          # <- each KV head serves `reps` query heads
    V = np.repeat(V, reps, axis=0)
    W = softmax(Q @ K.transpose(0,2,1) / np.sqrt(d_head))
    return (W @ V).transpose(1,0,2).reshape(n, -1)

for n_kv in [8, 4, 2, 1]:
    out = grouped_query_attention(X, n_q_heads=8, n_kv_heads=n_kv, d_model=32)
    label = "MHA" if n_kv==8 else "MQA" if n_kv==1 else "GQA"
    print(f"{label:3s}  KV heads={n_kv}  output {out.shape}  KV cache {8/n_kv:.0f}x smaller than MHA")

## Your turn 🧪
1. For a 70B-scale model (80 layers, 64 heads, `d_head=128`, 8k context), compute the MHA vs GQA-8 cache. How many GB
   do you save?
2. In `multi_head_attention`, set `h=1`. Is the output the same as single-head attention? Why is `h=1` weaker?
3. Add a **causal mask** (Lab 4) inside `multi_head_attention` — that turns it into exactly what GPT uses.

In [ ]:
# Your turn: 70B-scale cache
print("MHA :", kv_cache_bytes(8192, 64, 128, n_layers=80)/1e9, "GB")
print("GQA8:", kv_cache_bytes(8192,  8, 128, n_layers=80)/1e9, "GB")